In [ ]:
import os
import glob
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from pytorch_wavelets import DWTForward
from tqdm import tqdm
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------------------------------------------------
# Dataset
# -------------------------------------------------------------------------
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, size=256):
        self.images = sorted(glob.glob(os.path.join(image_dir, "*")))
        self.masks  = sorted(glob.glob(os.path.join(mask_dir, "*")))
        self.timg = T.Compose([T.Resize((size,size)), T.ToTensor()])
        self.tmask = T.Compose([T.Resize((size,size)), T.ToTensor()])

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img  = Image.open(self.images[idx]).convert("RGB")
        mask = Image.open(self.masks[idx]).convert("L")
        img  = self.timg(img)
        mask = (self.tmask(mask) > 0.5).float()
        return img, mask

In [ ]:
# -------------------------------------------------------------------------
# Wavelet-EAM (db6)
# -------------------------------------------------------------------------
class WEAM(nn.Module):
    def __init__(self, channels, wave='db6', lambda_val=1e-4):
        super().__init__()
        self.C = channels
        self.lambda_val = lambda_val

        self.dwt = DWTForward(J=1, wave=wave, mode='reflect')
        self.bn = nn.BatchNorm2d(channels)
        self.reduction = nn.Conv2d(channels*4, channels*4, 1)

    def spatial_attention(self, x):
        mean = x.mean([2,3], keepdim=True)
        var  = ((x-mean)**2).mean([2,3], keepdim=True)
        energy = 4*(var+self.lambda_val)*(x-mean)**2 + 2*var + 2*self.lambda_val
        return 1/(energy+1e-6)

    def frequency_attention(self, x):
        mag = torch.abs(x)
        return mag / (mag.mean([2,3], keepdim=True) + 1e-6)

    def forward(self, x):
        B,C,H,W = x.shape

        LL, HF = self.dwt(x)
        LH, HL, HH = HF[0]

        # Resize to original
        LL = F.interpolate(LL,(H,W))
        LH = F.interpolate(LH,(H,W))
        HL = F.interpolate(HL,(H,W))
        HH = F.interpolate(HH,(H,W))

        gamma = self.bn.weight.abs()
        gamma = gamma / gamma.sum()
        gamma = gamma.view(1,C,1,1)

        LL *= gamma; LH *= gamma; HL *= gamma; HH *= gamma

        LL = torch.sigmoid(self.spatial_attention(LL)) * LL
        LH = torch.sigmoid(self.frequency_attention(LH)) * LH
        HL = torch.sigmoid(self.frequency_attention(HL)) * HL
        HH = torch.sigmoid(self.frequency_attention(HH)) * HH

        out = torch.cat([LL,LH,HL,HH], dim=1)
        return self.reduction(out)

In [ ]:
# -------------------------------------------------------------------------
# EDDC block
# -------------------------------------------------------------------------
class EDDC(nn.Module):
    def __init__(self, channels, sigma_x=1.0, sigma_y=1.8):
        super().__init__()

        yy, xx = torch.meshgrid(torch.arange(5), torch.arange(5))
        yy = yy - 2
        xx = xx - 2

        mask = torch.exp(-((xx**2)/(2*sigma_x**2) + (yy**2)/(2*sigma_y**2)))
        mask = mask / mask.max()
        mask = mask.view(1,1,5,5)

        self.register_buffer("mask", mask)

        self.dw = nn.Conv2d(channels, channels, 5, padding=2, groups=channels, bias=False)
        self.pw = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        w = self.dw.weight * self.mask
        return x + self.pw(self.dw(x))

# -------------------------------------------------------------------------
# PAC block
# -------------------------------------------------------------------------
class PAC(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.radial = nn.Conv2d(channels, channels, 1)
        self.angular = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B,C,H,W = x.shape

        yy, xx = torch.meshgrid(torch.arange(H), torch.arange(W), indexing='ij')
        yy = yy.to(x.device) - H/2
        xx = xx.to(x.device) - W/2

        r = torch.sqrt(xx**2 + yy**2); r = r / r.max()
        t = torch.atan2(yy,xx); t = (t + math.pi)/(2*math.pi)

        r = r.unsqueeze(0).unsqueeze(0)
        t = t.unsqueeze(0).unsqueeze(0)

        return x + self.radial(x*r) + self.angular(x*t)


In [ ]:
# -------------------------------------------------------------------------
# Basic ConvBlock
# -------------------------------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self,in_ch,out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,3,padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch,out_ch,3,padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
        )
    def forward(self,x):
        return self.conv(x)

# -------------------------------------------------------------------------
# HEA-Net-WEP
# -------------------------------------------------------------------------
class HEANetWEP(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, bottleneck_mode="EDDC"):
        super().__init__()

        self.enc1 = ConvBlock(in_channels,64)
        self.weam1 = WEAM(64)

        self.enc2 = ConvBlock(64*4,128)
        self.weam2 = WEAM(128)

        self.enc3 = ConvBlock(128*4,256)
        self.weam3 = WEAM(256)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(256*4,512)
        self.mixer = EDDC(512) if bottleneck_mode=="EDDC" else PAC(512)

        self.up3 = nn.ConvTranspose2d(512,256,2,2)
        self.dec3 = ConvBlock(256 + 256*4, 256)

        self.up2 = nn.ConvTranspose2d(256,128,2,2)
        self.dec2 = ConvBlock(128 + 128*4, 128)

        self.up1 = nn.ConvTranspose2d(128,64,2,2)
        self.dec1 = ConvBlock(64 + 64*4, 64)

        self.final = nn.Conv2d(64,out_channels,1)

    def forward(self,x):
        e1 = self.weam1(self.enc1(x))
        e2 = self.weam2(self.enc2(self.pool(e1)))
        e3 = self.weam3(self.enc3(self.pool(e2)))

        b  = self.mixer(self.bottleneck(self.pool(e3)))

        d3 = self.dec3(torch.cat([self.up3(b), e3],1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2],1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1],1))

        return torch.sigmoid(self.final(d1))

In [ ]:
# -------------------------------------------------------------------------
# Loss + Metrics
# -------------------------------------------------------------------------
def dice_loss(pred,target,eps=1e-6):
    inter = (pred*target).sum()
    return 1 - (2*inter+eps)/(pred.sum()+target.sum()+eps)

def combined_loss(pred,target):
    return F.binary_cross_entropy(pred,target) + dice_loss(pred,target)

def compute_metrics(pred,target):
    p = (pred>0.5).float()
    inter = (p*target).sum()
    dice = (2*inter)/(p.sum()+target.sum()+1e-6)
    iou  = inter/(p.sum()+target.sum()-inter+1e-6)
    return dice.item(),iou.item()

In [ ]:
# -------------------------------------------------------------------------
# Training Loop
# -------------------------------------------------------------------------
def train_model(train_loader,val_loader,epochs=50,lr=1e-3,bottleneck="EDDC"):
    model = HEANetWEP(bottleneck_mode=bottleneck).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for img,mask in tqdm(train_loader):
            img,mask = img.to(device), mask.to(device)
            pred = model(img)
            loss = combined_loss(pred,mask)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        model.eval()
        dice, iou = 0,0
        with torch.no_grad():
            for img,mask in val_loader:
                img,mask = img.to(device), mask.to(device)
                pred = model(img)
                d,i = compute_metrics(pred,mask)
                dice+=d; iou+=i

        print(f"Epoch {epoch+1}/{epochs} | Loss={total_loss/len(train_loader):.4f} | Dice={dice/len(val_loader):.4f} | IoU={iou/len(val_loader):.4f}")

    return model

In [ ]:
# -------------------------------------------------------------------------
# Run training
# -------------------------------------------------------------------------
train_images = "/home/nit/Lopa/Segmentation/Monuseg_segmentation/MonuSeg/Training/TissueImages"
train_masks  = "/home/nit/Lopa/Segmentation/Monuseg_segmentation/MonuSeg/Training/GroundTruth"
val_images   = "/home/nit/Lopa/Segmentation/Monuseg_segmentation/MonuSeg/Test/TissueImages"
val_masks    = "/home/nit/Lopa/Segmentation/Monuseg_segmentation/MonuSeg/Test/GroundTruth"

train_loader = DataLoader(SegmentationDataset(train_images,train_masks),batch_size=4,shuffle=True)
val_loader   = DataLoader(SegmentationDataset(val_images,val_masks),  batch_size=4,shuffle=False)

model = train_model(train_loader,val_loader,epochs=100,bottleneck="EDDC")